In [1]:
# Deescuy Advanced Pipeline
# Dibuat mengikuti insight EDA:
# - data 2 row per match
# - temporal split wajib
# - train-test shift nyata, terutama gender dan era
# - banyak fitur history train tidak ada di test
# - perlu feature engineering historis leakage-safe
# - perlu inferensi test secara rekursif

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from collections import defaultdict, deque
import copy
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("/mnt/data")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = DATA_DIR / "submission.csv"

FIT_START_YEAR = 1980
VAL_WINDOWS = [(2004, 2005), (2006, 2007), (2008, 2009), (2010, 2011)]
MAX_GOALS_CLIP = 12.0
BLEND_MAIN = 0.65
BLEND_SPEC = 0.20
BLEND_PRIOR = 0.15


def safe_mean(values, default=np.nan):
    vals = [v for v in values if pd.notna(v)]
    return float(np.mean(vals)) if len(vals) else default


def weighted_nanmean(values, weights, default=np.nan):
    pairs = [(v, w) for v, w in zip(values, weights) if pd.notna(v) and pd.notna(w)]
    if len(pairs) == 0:
        return default
    vals = np.array([p[0] for p in pairs], dtype=float)
    wts = np.array([p[1] for p in pairs], dtype=float)
    if np.isclose(wts.sum(), 0):
        return float(vals.mean())
    return float(np.average(vals, weights=wts))


def clean_altitude(x):
    if pd.isna(x):
        return np.nan
    return np.nan if x == -9999 else float(x)


def safe_log_diff(a, b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return float(np.log1p(a) - np.log1p(b))


def make_match_frame(df, is_train=True):
    home = df[df["is_home"] == 1].copy()
    away = df[df["is_home"] == 0].copy()

    base_cols = [
        "match_id", "Id", "team", "opponent", "neutral", "tournament", "venue_country",
        "confederation_team", "confederation_opp",
        "population_team", "population_opp",
        "gdp_per_capita_team", "gdp_per_capita_opp",
        "altitude_venue", "distance_travel_team", "distance_travel_opp", "temperature_venue",
        "date", "gender"
    ]

    h = home[base_cols + (["team_goals", "opp_goals"] if is_train else [])].copy()
    a = away[base_cols + (["team_goals", "opp_goals"] if is_train else [])].copy()
    m = h.merge(a, on="match_id", suffixes=("_home_row", "_away_row"))

    out = pd.DataFrame({
        "match_id": m["match_id"],
        "date": m["date_home_row"],
        "gender": m["gender_home_row"],
        "neutral": m["neutral_home_row"],
        "tournament": m["tournament_home_row"],
        "venue_country": m["venue_country_home_row"],
        "home_id": m["Id_home_row"],
        "away_id": m["Id_away_row"],
        "home_team": m["team_home_row"],
        "away_team": m["team_away_row"],
        "home_confed": m["confederation_team_home_row"],
        "away_confed": m["confederation_team_away_row"],
        "home_population": m["population_team_home_row"],
        "away_population": m["population_team_away_row"],
        "home_gdp_pc": m["gdp_per_capita_team_home_row"],
        "away_gdp_pc": m["gdp_per_capita_team_away_row"],
        "altitude_venue": m["altitude_venue_home_row"],
        "home_distance_travel": m["distance_travel_team_home_row"],
        "away_distance_travel": m["distance_travel_team_away_row"],
        "temperature_venue": m["temperature_venue_home_row"],
    })

    if is_train:
        out["home_goals"] = m["team_goals_home_row"]
        out["away_goals"] = m["team_goals_away_row"]

    return out.sort_values(["date", "match_id"]).reset_index(drop=True)


def make_submission_from_match_predictions(test_match, pred_home, pred_away):
    pred_df = test_match[["match_id", "home_id", "away_id"]].copy()
    pred_df["pred_home_goals"] = np.asarray(pred_home, dtype=float)
    pred_df["pred_away_goals"] = np.asarray(pred_away, dtype=float)

    home_rows = pred_df[["home_id", "pred_home_goals", "pred_away_goals"]].rename(columns={
        "home_id": "Id",
        "pred_home_goals": "team_goals",
        "pred_away_goals": "opp_goals",
    })
    away_rows = pred_df[["away_id", "pred_home_goals", "pred_away_goals"]].rename(columns={
        "away_id": "Id",
        "pred_home_goals": "opp_goals",
        "pred_away_goals": "team_goals",
    })

    submission = pd.concat([home_rows, away_rows], axis=0, ignore_index=True)
    return submission[["Id", "team_goals", "opp_goals"]].sort_values("Id").reset_index(drop=True)


def make_empty_team_state():
    return {
        "matches": 0,
        "goals_for": 0.0,
        "goals_against": 0.0,
        "points": 0.0,
        "wins": 0.0,
        "draws": 0.0,
        "gf5": deque(maxlen=5),
        "ga5": deque(maxlen=5),
        "pts5": deque(maxlen=5),
        "gf10": deque(maxlen=10),
        "ga10": deque(maxlen=10),
        "pts10": deque(maxlen=10),
        "last_date": None,
        "elo": 1500.0,
        "home_matches": 0,
        "home_goals_for": 0.0,
        "home_goals_against": 0.0,
        "away_matches": 0,
        "away_goals_for": 0.0,
        "away_goals_against": 0.0,
    }


def make_empty_pair_state():
    return {
        "matches": 0,
        "goals_for": 0.0,
        "goals_against": 0.0,
        "points": 0.0,
        "gf5": deque(maxlen=5),
        "ga5": deque(maxlen=5),
        "pts5": deque(maxlen=5),
    }


def make_empty_group_state():
    return {
        "matches": 0,
        "home_goals": 0.0,
        "away_goals": 0.0,
        "total_goals": 0.0,
        "draws": 0.0,
    }


def init_states():
    return {
        "team": defaultdict(make_empty_team_state),
        "pair": defaultdict(make_empty_pair_state),
        "tournament": defaultdict(make_empty_group_state),
        "confed": defaultdict(make_empty_group_state),
        "gender": defaultdict(make_empty_group_state),
        "global": make_empty_group_state(),
    }


def get_group_avg(st, part="total"):
    if st["matches"] == 0:
        return np.nan
    if part == "home":
        return st["home_goals"] / st["matches"]
    if part == "away":
        return st["away_goals"] / st["matches"]
    if part == "draw":
        return st["draws"] / st["matches"]
    return st["total_goals"] / st["matches"]


def get_team_features(states, key, side, current_date):
    st = states["team"][key]
    d = {
        f"{side}_matches_pre": st["matches"],
        f"{side}_gf_avg_pre": st["goals_for"] / st["matches"] if st["matches"] else np.nan,
        f"{side}_ga_avg_pre": st["goals_against"] / st["matches"] if st["matches"] else np.nan,
        f"{side}_pts_avg_pre": st["points"] / st["matches"] if st["matches"] else np.nan,
        f"{side}_win_rate_pre": st["wins"] / st["matches"] if st["matches"] else np.nan,
        f"{side}_draw_rate_pre": st["draws"] / st["matches"] if st["matches"] else np.nan,
        f"{side}_gf_last5_pre": safe_mean(st["gf5"]),
        f"{side}_ga_last5_pre": safe_mean(st["ga5"]),
        f"{side}_pts_last5_pre": safe_mean(st["pts5"]),
        f"{side}_gf_last10_pre": safe_mean(st["gf10"]),
        f"{side}_ga_last10_pre": safe_mean(st["ga10"]),
        f"{side}_pts_last10_pre": safe_mean(st["pts10"]),
        f"{side}_elo_pre": st["elo"],
        f"{side}_home_gf_avg_pre": st["home_goals_for"] / st["home_matches"] if st["home_matches"] else np.nan,
        f"{side}_home_ga_avg_pre": st["home_goals_against"] / st["home_matches"] if st["home_matches"] else np.nan,
        f"{side}_away_gf_avg_pre": st["away_goals_for"] / st["away_matches"] if st["away_matches"] else np.nan,
        f"{side}_away_ga_avg_pre": st["away_goals_against"] / st["away_matches"] if st["away_matches"] else np.nan,
    }
    d[f"{side}_days_since_pre"] = np.nan if st["last_date"] is None else (current_date - st["last_date"]).days
    return d


def compute_prior_goals(feat):
    global_home = feat.get("global_home_goals_avg_pre", np.nan)
    global_away = feat.get("global_away_goals_avg_pre", np.nan)

    prior_home = weighted_nanmean(
        [
            feat.get("home_gf_last5_pre"), feat.get("home_gf_avg_pre"),
            feat.get("away_ga_last5_pre"), feat.get("away_ga_avg_pre"),
            feat.get("h2h_gf_avg_pre"), feat.get("tourn_home_goals_avg_pre"),
            feat.get("confed_home_goals_avg_pre"), feat.get("gender_home_goals_avg_pre"),
            global_home,
        ],
        [0.24, 0.18, 0.16, 0.12, 0.08, 0.08, 0.05, 0.04, 0.05],
        default=global_home,
    )

    prior_away = weighted_nanmean(
        [
            feat.get("away_gf_last5_pre"), feat.get("away_gf_avg_pre"),
            feat.get("home_ga_last5_pre"), feat.get("home_ga_avg_pre"),
            feat.get("h2h_ga_avg_pre"), feat.get("tourn_away_goals_avg_pre"),
            feat.get("confed_away_goals_avg_pre"), feat.get("gender_away_goals_avg_pre"),
            global_away,
        ],
        [0.24, 0.18, 0.16, 0.12, 0.08, 0.08, 0.05, 0.04, 0.05],
        default=global_away,
    )

    return prior_home, prior_away


def compute_match_features(row, states):
    date = row.date
    gender = row.gender

    home_key = (gender, row.home_team)
    away_key = (gender, row.away_team)
    pair_key = (gender, row.home_team, row.away_team)
    tourn_key = (gender, row.tournament)
    confed_key = (gender, row.home_confed, row.away_confed)
    gender_key = (gender,)

    feat = {
        "match_id": row.match_id,
        "date": row.date,
        "year": row.date.year,
        "month": row.date.month,
        "dayofyear": row.date.dayofyear,
        "weekday": row.date.dayofweek,
        "is_weekend": int(row.date.dayofweek >= 5),
        "gender": row.gender,
        "neutral": row.neutral,
        "tournament": row.tournament,
        "venue_country": row.venue_country,
        "home_team": row.home_team,
        "away_team": row.away_team,
        "home_confed": row.home_confed,
        "away_confed": row.away_confed,
        "same_confed": int(row.home_confed == row.away_confed),
        "altitude_venue_clean": clean_altitude(row.altitude_venue),
        "temperature_venue": row.temperature_venue,
        "home_distance_travel": row.home_distance_travel,
        "away_distance_travel": row.away_distance_travel,
        "travel_diff": row.home_distance_travel - row.away_distance_travel if pd.notna(row.home_distance_travel) and pd.notna(row.away_distance_travel) else np.nan,
        "travel_sum": row.home_distance_travel + row.away_distance_travel if pd.notna(row.home_distance_travel) and pd.notna(row.away_distance_travel) else np.nan,
        "home_population": row.home_population,
        "away_population": row.away_population,
        "home_gdp_pc": row.home_gdp_pc,
        "away_gdp_pc": row.away_gdp_pc,
        "log_pop_diff": safe_log_diff(row.home_population, row.away_population),
        "log_gdp_diff": safe_log_diff(row.home_gdp_pc, row.away_gdp_pc),
        "venue_is_home_country": int(isinstance(row.venue_country, str) and row.venue_country == row.home_team),
        "venue_is_away_country": int(isinstance(row.venue_country, str) and row.venue_country == row.away_team),
    }

    feat.update(get_team_features(states, home_key, "home", date))
    feat.update(get_team_features(states, away_key, "away", date))

    pair_state = states["pair"][pair_key]
    feat.update({
        "h2h_matches_pre": pair_state["matches"],
        "h2h_gf_avg_pre": pair_state["goals_for"] / pair_state["matches"] if pair_state["matches"] else np.nan,
        "h2h_ga_avg_pre": pair_state["goals_against"] / pair_state["matches"] if pair_state["matches"] else np.nan,
        "h2h_pts_avg_pre": pair_state["points"] / pair_state["matches"] if pair_state["matches"] else np.nan,
        "h2h_gf_last5_pre": safe_mean(pair_state["gf5"]),
        "h2h_ga_last5_pre": safe_mean(pair_state["ga5"]),
        "h2h_pts_last5_pre": safe_mean(pair_state["pts5"]),
    })

    tourn_state = states["tournament"][tourn_key]
    feat.update({
        "tourn_matches_pre": tourn_state["matches"],
        "tourn_home_goals_avg_pre": get_group_avg(tourn_state, "home"),
        "tourn_away_goals_avg_pre": get_group_avg(tourn_state, "away"),
        "tourn_total_goals_avg_pre": get_group_avg(tourn_state, "total"),
        "tourn_draw_rate_pre": get_group_avg(tourn_state, "draw"),
    })

    confed_state = states["confed"][confed_key]
    feat.update({
        "confed_matches_pre": confed_state["matches"],
        "confed_home_goals_avg_pre": get_group_avg(confed_state, "home"),
        "confed_away_goals_avg_pre": get_group_avg(confed_state, "away"),
        "confed_total_goals_avg_pre": get_group_avg(confed_state, "total"),
        "confed_draw_rate_pre": get_group_avg(confed_state, "draw"),
    })

    gender_state = states["gender"][gender_key]
    feat.update({
        "gender_matches_pre": gender_state["matches"],
        "gender_home_goals_avg_pre": get_group_avg(gender_state, "home"),
        "gender_away_goals_avg_pre": get_group_avg(gender_state, "away"),
        "gender_total_goals_avg_pre": get_group_avg(gender_state, "total"),
        "gender_draw_rate_pre": get_group_avg(gender_state, "draw"),
    })

    global_state = states["global"]
    feat.update({
        "global_matches_pre": global_state["matches"],
        "global_home_goals_avg_pre": get_group_avg(global_state, "home"),
        "global_away_goals_avg_pre": get_group_avg(global_state, "away"),
        "global_total_goals_avg_pre": get_group_avg(global_state, "total"),
        "global_draw_rate_pre": get_group_avg(global_state, "draw"),
    })

    diff_bases = [
        "matches_pre", "gf_avg_pre", "ga_avg_pre", "pts_avg_pre",
        "win_rate_pre", "draw_rate_pre", "gf_last5_pre", "ga_last5_pre", "pts_last5_pre",
        "gf_last10_pre", "ga_last10_pre", "pts_last10_pre", "elo_pre", "days_since_pre",
        "home_gf_avg_pre", "home_ga_avg_pre", "away_gf_avg_pre", "away_ga_avg_pre",
    ]
    for base in diff_bases:
        h = feat.get(f"home_{base}", np.nan)
        a = feat.get(f"away_{base}", np.nan)
        feat[f"{base}_diff"] = h - a if pd.notna(h) and pd.notna(a) else np.nan

    prior_home, prior_away = compute_prior_goals(feat)
    feat["prior_home_goals"] = prior_home
    feat["prior_away_goals"] = prior_away
    feat["prior_total_goals"] = prior_home + prior_away if pd.notna(prior_home) and pd.notna(prior_away) else np.nan
    feat["prior_goal_diff"] = prior_home - prior_away if pd.notna(prior_home) and pd.notna(prior_away) else np.nan
    return feat


def update_team_state(st, date, gf, ga, side):
    pts = 3 if gf > ga else 1 if gf == ga else 0
    st["matches"] += 1
    st["goals_for"] += gf
    st["goals_against"] += ga
    st["points"] += pts
    st["wins"] += int(gf > ga)
    st["draws"] += int(gf == ga)
    st["gf5"].append(gf)
    st["ga5"].append(ga)
    st["pts5"].append(pts)
    st["gf10"].append(gf)
    st["ga10"].append(ga)
    st["pts10"].append(pts)
    st["last_date"] = date
    if side == "home":
        st["home_matches"] += 1
        st["home_goals_for"] += gf
        st["home_goals_against"] += ga
    else:
        st["away_matches"] += 1
        st["away_goals_for"] += gf
        st["away_goals_against"] += ga


def update_group_state(st, hg, ag):
    st["matches"] += 1
    st["home_goals"] += hg
    st["away_goals"] += ag
    st["total_goals"] += (hg + ag)
    st["draws"] += int(hg == ag)


def update_pair_state(st, gf, ga):
    pts = 3 if gf > ga else 1 if gf == ga else 0
    st["matches"] += 1
    st["goals_for"] += gf
    st["goals_against"] += ga
    st["points"] += pts
    st["gf5"].append(gf)
    st["ga5"].append(ga)
    st["pts5"].append(pts)


def update_elo(states, home_key, away_key, home_goals, away_goals, k=24):
    home_elo = states["team"][home_key]["elo"]
    away_elo = states["team"][away_key]["elo"]
    exp_home = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))
    act_home = 1.0 if home_goals > away_goals else 0.5 if home_goals == away_goals else 0.0
    act_away = 1.0 - act_home
    states["team"][home_key]["elo"] += k * (act_home - exp_home)
    states["team"][away_key]["elo"] += k * (act_away - (1 - exp_home))

    goal_diff = abs(home_goals - away_goals)
    if goal_diff >= 2:
        bonus = min(5.0, float(goal_diff - 1))
        if home_goals > away_goals:
            states["team"][home_key]["elo"] += bonus
            states["team"][away_key]["elo"] -= bonus
        elif away_goals > home_goals:
            states["team"][home_key]["elo"] -= bonus
            states["team"][away_key]["elo"] += bonus


def update_states(states, row, home_goals, away_goals):
    gender = row.gender
    date = row.date
    home_key = (gender, row.home_team)
    away_key = (gender, row.away_team)
    pair_home = (gender, row.home_team, row.away_team)
    pair_away = (gender, row.away_team, row.home_team)
    tourn_key = (gender, row.tournament)
    confed_key = (gender, row.home_confed, row.away_confed)
    gender_key = (gender,)

    update_team_state(states["team"][home_key], date, home_goals, away_goals, "home")
    update_team_state(states["team"][away_key], date, away_goals, home_goals, "away")
    update_pair_state(states["pair"][pair_home], home_goals, away_goals)
    update_pair_state(states["pair"][pair_away], away_goals, home_goals)
    update_group_state(states["tournament"][tourn_key], home_goals, away_goals)
    update_group_state(states["confed"][confed_key], home_goals, away_goals)
    update_group_state(states["gender"][gender_key], home_goals, away_goals)
    update_group_state(states["global"], home_goals, away_goals)
    update_elo(states, home_key, away_key, home_goals, away_goals)


def build_train_features(train_match):
    states = init_states()
    rows = []
    for row in train_match.itertuples(index=False):
        feat = compute_match_features(row, states)
        rows.append(feat)
        update_states(states, row, float(row.home_goals), float(row.away_goals))
    return pd.DataFrame(rows), states


def make_sample_weights(feat_df, train_gender_dist, test_gender_dist):
    year_norm = (feat_df["year"] - feat_df["year"].min()) / (feat_df["year"].max() - feat_df["year"].min())
    recency_weight = 0.35 + 0.65 * np.exp(2.2 * year_norm) / np.exp(2.2)
    gender_weight_map = {}
    for g in sorted(set(train_gender_dist) | set(test_gender_dist)):
        tr = train_gender_dist.get(g, 1e-6)
        te = test_gender_dist.get(g, 1e-6)
        gender_weight_map[g] = te / max(tr, 1e-6)
    gender_weight = feat_df["gender"].map(gender_weight_map).astype(float)
    return recency_weight * gender_weight


def build_catboost_model():
    return CatBoostRegressor(
        loss_function="Poisson",
        eval_metric="MAE",
        learning_rate=0.035,
        depth=8,
        l2_leaf_reg=8,
        random_strength=0.5,
        bootstrap_type="Bernoulli",
        subsample=0.85,
        min_data_in_leaf=20,
        iterations=2500,
        early_stopping_rounds=200,
        verbose=False,
        random_seed=SEED,
        allow_writing_files=False,
    )


def blend_predictions(main_pred, specialist_pred, prior_pred):
    if specialist_pred is None:
        return np.clip((0.85 * main_pred) + (0.15 * prior_pred), 0, MAX_GOALS_CLIP)
    return np.clip(
        (BLEND_MAIN * main_pred) + (BLEND_SPEC * specialist_pred) + (BLEND_PRIOR * prior_pred),
        0,
        MAX_GOALS_CLIP,
    )


def prepare_X(df, feature_cols, categorical_cols):
    X = df[feature_cols].copy()
    for c in categorical_cols:
        X[c] = X[c].astype(str).fillna("__MISSING__")
    return X


def fit_target_bundle(train_feat, train_match, target_col, feature_cols, categorical_cols, train_gender_dist, test_gender_dist):
    X_all = prepare_X(train_feat, feature_cols, categorical_cols)
    y_all = train_match[target_col].values
    w_all = make_sample_weights(train_feat, train_gender_dist, test_gender_dist)

    oof = np.full(len(train_feat), np.nan, dtype=float)
    fold_rows = []

    for val_start, val_end in VAL_WINDOWS:
        tr_mask = (train_feat["year"] < val_start) & (train_feat["year"] >= FIT_START_YEAR)
        va_mask = (train_feat["year"] >= val_start) & (train_feat["year"] <= val_end)
        tr_idx = train_feat.index[tr_mask]
        va_idx = train_feat.index[va_mask]
        if len(tr_idx) == 0 or len(va_idx) == 0:
            continue

        main_model = build_catboost_model()
        main_model.fit(
            X_all.loc[tr_idx], y_all[tr_idx],
            sample_weight=w_all[tr_idx],
            eval_set=(X_all.loc[va_idx], y_all[va_idx]),
            cat_features=categorical_cols,
            use_best_model=True,
        )

        specialist_models = {}
        for g in sorted(train_feat.loc[tr_idx, "gender"].dropna().unique()):
            tr_g_mask = tr_mask & (train_feat["gender"] == g)
            va_g_mask = va_mask & (train_feat["gender"] == g)
            tr_g_idx = train_feat.index[tr_g_mask]
            va_g_idx = train_feat.index[va_g_mask]
            if len(tr_g_idx) < 300:
                continue
            specialist = build_catboost_model()
            if len(va_g_idx) > 0:
                specialist.fit(
                    X_all.loc[tr_g_idx], y_all[tr_g_idx],
                    sample_weight=w_all[tr_g_idx],
                    eval_set=(X_all.loc[va_g_idx], y_all[va_g_idx]),
                    cat_features=categorical_cols,
                    use_best_model=True,
                )
            else:
                specialist.fit(
                    X_all.loc[tr_g_idx], y_all[tr_g_idx],
                    sample_weight=w_all[tr_g_idx],
                    cat_features=categorical_cols,
                    use_best_model=False,
                )
            specialist_models[g] = specialist

        main_pred = np.clip(main_model.predict(X_all.loc[va_idx]), 0, MAX_GOALS_CLIP)
        specialist_pred = np.full(len(va_idx), np.nan, dtype=float)
        va_gender = train_feat.loc[va_idx, "gender"].values
        for g, model in specialist_models.items():
            mask = (va_gender == g)
            if mask.sum() == 0:
                continue
            specialist_pred[mask] = np.clip(model.predict(X_all.loc[va_idx[mask]]), 0, MAX_GOALS_CLIP)

        short = target_col.split("_")[0]
        prior_pred = train_feat.loc[va_idx, f"prior_{short}_goals"].values.astype(float)
        prior_fill = np.nanmedian(prior_pred)
        prior_pred = np.where(pd.isna(prior_pred), prior_fill, prior_pred)
        prior_pred = np.clip(prior_pred, 0, MAX_GOALS_CLIP)

        final_pred = []
        for i in range(len(va_idx)):
            spec = None if pd.isna(specialist_pred[i]) else specialist_pred[i]
            final_pred.append(blend_predictions(main_pred[i], spec, prior_pred[i]))
        final_pred = np.asarray(final_pred, dtype=float)

        oof[va_idx] = final_pred
        fold_rows.append({
            "target": target_col,
            "val_start": val_start,
            "val_end": val_end,
            "n_train": len(tr_idx),
            "n_valid": len(va_idx),
            "mae": mean_absolute_error(y_all[va_idx], final_pred),
            "main_best_iter": main_model.tree_count_,
        })

    final_mask = train_feat["year"] >= FIT_START_YEAR
    final_idx = train_feat.index[final_mask]

    final_main = build_catboost_model()
    final_main.fit(
        X_all.loc[final_idx], y_all[final_idx],
        sample_weight=w_all[final_idx],
        cat_features=categorical_cols,
        use_best_model=False,
    )

    final_specialists = {}
    for g in sorted(train_feat.loc[final_idx, "gender"].dropna().unique()):
        idx = train_feat.index[final_mask & (train_feat["gender"] == g)]
        if len(idx) < 300:
            continue
        specialist = build_catboost_model()
        specialist.fit(
            X_all.loc[idx], y_all[idx],
            sample_weight=w_all[idx],
            cat_features=categorical_cols,
            use_best_model=False,
        )
        final_specialists[g] = specialist

    return {
        "target_col": target_col,
        "oof": oof,
        "fold_scores": pd.DataFrame(fold_rows),
        "main_model": final_main,
        "specialist_models": final_specialists,
    }


def predict_with_bundle(bundle, feat_df, target_short, feature_cols, categorical_cols):
    X = prepare_X(feat_df, feature_cols, categorical_cols)
    main_pred = np.clip(bundle["main_model"].predict(X), 0, MAX_GOALS_CLIP)
    specialist_pred = np.full(len(feat_df), np.nan, dtype=float)
    genders = feat_df["gender"].values
    for g, model in bundle["specialist_models"].items():
        mask = (genders == g)
        if mask.sum() == 0:
            continue
        specialist_pred[mask] = np.clip(model.predict(X.loc[mask]), 0, MAX_GOALS_CLIP)
    prior_pred = feat_df[f"prior_{target_short}_goals"].values.astype(float)
    prior_fill = np.nanmedian(prior_pred)
    prior_pred = np.where(pd.isna(prior_pred), prior_fill, prior_pred)
    prior_pred = np.clip(prior_pred, 0, MAX_GOALS_CLIP)
    final_pred = []
    for i in range(len(feat_df)):
        spec = None if pd.isna(specialist_pred[i]) else specialist_pred[i]
        final_pred.append(blend_predictions(main_pred[i], spec, prior_pred[i]))
    return np.asarray(final_pred, dtype=float)


def predict_test_recursive(test_match, base_states, home_bundle, away_bundle, feature_cols, categorical_cols):
    states = copy.deepcopy(base_states)
    feat_rows = []
    pred_home = []
    pred_away = []

    for row in test_match.itertuples(index=False):
        feat = compute_match_features(row, states)
        feat_df = pd.DataFrame([feat])
        ph = float(predict_with_bundle(home_bundle, feat_df, "home", feature_cols, categorical_cols)[0])
        pa = float(predict_with_bundle(away_bundle, feat_df, "away", feature_cols, categorical_cols)[0])
        pred_home.append(ph)
        pred_away.append(pa)
        feat_rows.append(feat)
        update_states(states, row, ph, pa)

    return pd.DataFrame(feat_rows), np.asarray(pred_home, dtype=float), np.asarray(pred_away, dtype=float), states


def main():
    assert TRAIN_PATH.exists(), f"train.csv tidak ditemukan di {TRAIN_PATH}"
    assert TEST_PATH.exists(), f"test.csv tidak ditemukan di {TEST_PATH}"

    train = pd.read_csv(TRAIN_PATH, parse_dates=["date"])
    test = pd.read_csv(TEST_PATH, parse_dates=["date"])

    assert train.groupby("match_id").size().eq(2).all()
    assert test.groupby("match_id").size().eq(2).all()
    assert train.groupby("match_id")["is_home"].sum().eq(1).all()
    assert test.groupby("match_id")["is_home"].sum().eq(1).all()

    train_match = make_match_frame(train, is_train=True)
    test_match = make_match_frame(test, is_train=False)

    train_feat, final_train_states = build_train_features(train_match)

    train_gender_dist = train_match["gender"].value_counts(normalize=True).to_dict()
    test_gender_dist = test_match["gender"].value_counts(normalize=True).to_dict()

    exclude_cols = {"match_id", "date"}
    categorical_cols = [
        "gender", "tournament", "venue_country",
        "home_team", "away_team", "home_confed", "away_confed",
    ]
    feature_cols = [c for c in train_feat.columns if c not in exclude_cols]

    home_bundle = fit_target_bundle(
        train_feat, train_match, "home_goals", feature_cols, categorical_cols,
        train_gender_dist, test_gender_dist,
    )
    away_bundle = fit_target_bundle(
        train_feat, train_match, "away_goals", feature_cols, categorical_cols,
        train_gender_dist, test_gender_dist,
    )

    valid_home = ~np.isnan(home_bundle["oof"])
    valid_away = ~np.isnan(away_bundle["oof"])
    home_oof_mae = mean_absolute_error(train_match.loc[valid_home, "home_goals"], home_bundle["oof"][valid_home])
    away_oof_mae = mean_absolute_error(train_match.loc[valid_away, "away_goals"], away_bundle["oof"][valid_away])

    print("OOF MAE home_goals :", round(home_oof_mae, 6))
    print("OOF MAE away_goals :", round(away_oof_mae, 6))
    print("Mean OOF MAE       :", round(float(np.mean([home_oof_mae, away_oof_mae])), 6))
    print()
    print(home_bundle["fold_scores"])
    print()
    print(away_bundle["fold_scores"])

    _, pred_home, pred_away, _ = predict_test_recursive(
        test_match, final_train_states, home_bundle, away_bundle, feature_cols, categorical_cols
    )

    submission = make_submission_from_match_predictions(test_match, pred_home, pred_away)
    submission = test[["Id"]].merge(submission, on="Id", how="left")
    assert submission["team_goals"].notna().all()
    assert submission["opp_goals"].notna().all()
    submission.to_csv(SUBMISSION_PATH, index=False)

    print()
    print("Submission saved to:", SUBMISSION_PATH)
    print(submission.head())
    print("Submission shape:", submission.shape)


In [ ]:
main()